In [1]:
pip install ultralytics

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 5.4 MB/s  0:00:00
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
    --------------------------------------- 0.5/40.2 MB ? eta -:--:--
   - -------------------------------------- 1.0/40.2 MB 5.6 MB/s eta 0:00:07
   -- ------------------------------------- 2.1/40.2 MB 4.5 MB/s eta 0:00:09
   -- ------------------------------------- 2.4/40.2 MB 4.8 MB/s eta 0:00:08
   ---- ----------------------------------- 4.7/40.2 MB 5.5 MB/s eta 0:00:07
   ----- ---------------------------------- 5.8/40.2 MB 5.4 MB/s eta 0:00:07
   ------ --------------------------------- 6.8/40.2 MB 5.4 MB/s eta 0:00:07
   -------- ------------------------------- 8.1/40.2 MB 5.5 MB/s eta 0:00:06
   --------- ------------------------------ 9.2/40.2 MB 5.5 MB/s eta 0:00:06
   ---------- -------------------

In [2]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\MazurenkoEV\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
# загружаем модели (в том числе предобученные)
model = YOLO("yolo_models/yolo11n.pt")
# model = YOLO("yolo11n-seg.pt")
# model = YOLO("yolo11n-pose.pt")
# model = YOLO("path/to/best.pt")   если нужно загрузить кастом

In [4]:
# Perform tracking with the model

# results = model.track("/content/way_.mp4", show=True, tracker="bytetrack.yaml")  # with ByteTrack

In [5]:
# results = model.track(source="/content/way_.mp4", conf=0.3, iou=0.5, show=True)

In [9]:
from collections import defaultdict

import cv2
import numpy as np

from ultralytics import YOLO

# Load the YOLO11 model
model = YOLO("yolo_models/yolo11n.pt")

# Open the video file
video_path = "videos/way.mp4"
cap = cv2.VideoCapture(video_path)

# Store the track history
track_history = defaultdict(lambda: [])

# Loop through the video frames
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()

    if success:
        # Run YOLO11 tracking on the frame, persisting tracks between frames
        result = model.track(frame, persist=True)[0]

        # Get the boxes and track IDs
        if result.boxes and result.boxes.id is not None:
            boxes = result.boxes.xywh.cpu()
            track_ids = result.boxes.id.int().cpu().tolist()

            # Visualize the result on the frame
            frame = result.plot()

            # Plot the tracks
            for box, track_id in zip(boxes, track_ids):
                x, y, w, h = box
                track = track_history[track_id]
                track.append((float(x), float(y)))  # x, y center point
                if len(track) > 30:  # retain 30 tracks for 30 frames
                    track.pop(0)

                # Draw the tracking lines
                points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                cv2.polylines(frame, [points], isClosed=False, color=(230, 230, 230), thickness=10)

        # Display the annotated frame
        cv2.imshow("YOLO11 Tracking", frame) # отключим визуализацию

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # Break the loop if the end of the video is reached
        break

# Release the video capture object and close the display window
cap.release()
cv2.destroyAllWindows()


0: 384x640 6 cars, 1 airplane, 15.9ms
Speed: 1.5ms preprocess, 15.9ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 cars, 1 airplane, 9.8ms
Speed: 1.7ms preprocess, 9.8ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 cars, 1 airplane, 9.6ms
Speed: 2.2ms preprocess, 9.6ms inference, 3.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 cars, 1 airplane, 11.6ms
Speed: 2.6ms preprocess, 11.6ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 cars, 1 airplane, 8.7ms
Speed: 1.6ms preprocess, 8.7ms inference, 3.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 cars, 1 airplane, 6.5ms
Speed: 1.5ms preprocess, 6.5ms inference, 2.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 cars, 1 airplane, 14.3ms
Speed: 2.9ms preprocess, 14.3ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 cars, 1 airplane, 12.2ms
Speed: 1.9ms prep

Оценка скорости объектов (неверная): удаляющиеся объекты движутся быстрее чем приближающиеся

In [12]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error reading video file"


w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("videos/speed_management.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))


speedestimator = solutions.SpeedEstimator(
    show=True,  # display the output
    model="yolo_models/yolo11n.pt",  # path to the YOLO11 model file.
    fps=fps,  # adjust speed based on frame per second
    # max_speed=120,  # cap speed to a max value (km/h) to avoid outliers
    # max_hist=5,  # minimum frames object tracked before computing speed
    # meter_per_pixel=0.05,  # highly depends on the camera configuration
    # classes=[0, 2],  # estimate speed of specific classes.
    # line_width=2,  # adjust the line width for bounding boxes
)

while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or processing is complete.")
        break

    results = speedestimator(im0)

    # print(results)  # access the output

    video_writer.write(results.plot_im)  # write the processed frame.

cap.release()
video_writer.release()
cv2.destroyAllWindows()

Ultralytics Solutions:  {'source': None, 'model': 'yolo_models/yolo11n.pt', 'classes': None, 'show_conf': True, 'show_labels': True, 'region': None, 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 29, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': True, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'half': False, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}


0: 720x1280 18.3ms, 6 car, 1 airplane
Speed: 78.7ms track, 18.3ms solution per image at shape (1, 3, 720, 1280)

1: 720x1280 2.6ms, 6 car, 1 airplane
Speed: 23.1ms track, 2.6ms solution per image at shape (1, 3, 720, 1280)

2: 720x1280 2.3ms, 6 car, 1 airplane
Speed: 20.3ms track, 2.3ms solution per image at shape (1, 3, 720, 1280)

3: 720x1280 2.2ms, 6 car, 1 airplane
Speed: 19.9ms track, 2.2ms solution per image at shape (1, 3, 720, 1280)

4: 720x1280 2.5ms, 6 car, 1 airplane
Speed: 18.6ms track, 2.5ms solution per image at shape (1, 3, 720, 1280)

5: 720x1280 2.5ms, 6 car, 1 airplane
Speed: 18.9ms track, 2.5ms solution per image at shape (1, 3, 720, 1280)

6: 720x1280 2.4ms, 6 car, 1 airplane
Speed: 18.8ms track, 2.4ms solution per image at shape (1, 3, 720, 1280)

7: 720x1280 2.4ms, 7 car, 1 airplane
Speed: 19.1ms track, 2.4ms solution per image at shape (1, 3, 720, 1280)

8: 720x1280 2.3ms, 8 car, 1 airplane
Speed: 18.2ms track, 2.3ms solution per image at shape (1, 3, 720, 1280)


Тепловая карта объектов

In [ ]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error reading video file"


w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("videos/heatmap_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# For object counting with heatmap, you can pass region points.
# region_points = [(20, 400), (1080, 400)]                                      # line points
# region_points = [(20, 400), (1080, 400), (1080, 360), (20, 360)]              # rectangle region
# region_points = [(20, 400), (1080, 400), (1080, 360), (20, 360), (20, 400)]   # polygon points


heatmap = solutions.Heatmap(
    show=True,  # display the output
    model="yolo_models/yolo11n.pt",  # path to the YOLO11 model file
    colormap=cv2.COLORMAP_PARULA,  # colormap of heatmap
    # region=region_points,  # object counting with heatmaps, you can pass region_points
    # classes=[0, 2],  # generate heatmap for specific classes i.e person and car.
)

while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or processing is complete.")
        break

    results = heatmap(im0)

    # print(results)  # access the output

    video_writer.write(results.plot_im)

cap.release()
video_writer.release()
cv2.destroyAllWindows()

Ultralytics Solutions:  {'source': None, 'model': 'yolo11n.pt', 'classes': None, 'show_conf': True, 'show_labels': True, 'region': None, 'colormap': 12, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': True, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'half': False, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}
0: 720x1280 41.8ms, 6 car, 1 airplane
Speed: 80.1ms track, 41.8ms solution per image at shape (1, 3, 720, 1280)

1: 720x1280 13.8ms, 6 car, 1 airplane
Speed: 27.4ms track, 13.8ms solution per image at shape (1, 3, 720, 1280)

2: 720x1280 14.6ms, 6 car, 1 airplane
Speed: 23.4ms track, 14.6ms solution per image at shape (1, 3, 720, 1280)

3: 720x1280 14.6m

Отслеживание объектов в зоне

In [ ]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error reading video file"

# Define region points
region_points = [(150, 150), (1130, 150), (1130, 570), (150, 570)]


w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("video/trackzone_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))


trackzone = solutions.TrackZone(
    show=True,  # display the output
    region=region_points,  # pass region points
    model="yolo_models/yolo11n.pt",  # use any model that Ultralytics support, i.e. YOLOv9, YOLOv10
    # line_width=2,  # adjust the line width for bounding boxes and text display
)

while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        print("Video frame is empty or processing is complete.")
        break

    results = trackzone(im0)

    # print(results)  # access the output

    video_writer.write(results.plot_im)

cap.release()
video_writer.release()
cv2.destroyAllWindows()

Ultralytics Solutions:  {'source': None, 'model': 'yolo11n.pt', 'classes': None, 'show_conf': True, 'show_labels': True, 'region': [(150, 150), (1130, 150), (1130, 570), (150, 570)], 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': True, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'half': False, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}
0: 720x1280 20.8ms, 5 car, 1 airplane
Speed: 84.7ms track, 20.8ms solution per image at shape (1, 3, 720, 1280)

1: 720x1280 7.5ms, 5 car, 1 airplane
Speed: 20.6ms track, 7.5ms solution per image at shape (1, 3, 720, 1280)

2: 720x1280 5.8ms, 5 car, 1 airplane
Speed: 32.4ms track, 5.8ms solution per image at 

Сегментация экземпляров с отслеживанием объектов

In [13]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error reading video file"

# Video writer
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("videos/isegment_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Initialize instance segmentation object
isegment = solutions.InstanceSegmentation(
    show=True,  # display the output
    model="yolo_models/yolo11n-seg.pt",  # model="yolo11n-seg.pt" for object segmentation using YOLO11.
    # classes=[0, 2],  # segment specific classes i.e, person and car with pretrained model.
)

# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    results = isegment(im0)

    # print(results)  # access the output

    video_writer.write(results.plot_im)

cap.release()
video_writer.release()
cv2.destroyAllWindows()

Ultralytics Solutions:  {'source': None, 'model': 'yolo_models/yolo11n-seg.pt', 'classes': None, 'show_conf': True, 'show_labels': True, 'region': None, 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': True, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'half': False, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}


0: 720x1280 60.1ms, 5 car, 1 airplane
Speed: 99.9ms track, 60.1ms solution per image at shape (1, 3, 720, 1280)

1: 720x1280 6.6ms, 5 car, 1 airplane
Speed: 50.0ms track, 6.6ms solution per image at shape (1, 3, 720, 1280)

2: 720x1280 5.5ms, 5 car, 1 airplane
Speed: 19.0ms track, 5.5ms solution per image at shape (1, 3, 720, 1280)

3: 720x1280 5.7ms, 5 car, 1 airplane
Speed: 20.9ms track, 5.7ms solution per image at shape (1, 3, 720, 1280)

4: 720x1280 6.1ms, 6 car, 1 airplane
Speed: 19.4ms track, 6.1ms solution per image at shape (1, 3, 720, 1280)

5: 720x1280 5.5ms, 6 car, 1 airplane
Speed: 19.5ms track, 5.5ms solution per image at shape (1, 3, 720, 1280)

6: 720x1280 5.6ms, 6 car, 1 airplane
Speed: 18.6ms track, 5.6ms solution per image at shape (1, 3, 720, 1280)

7: 720x1280 64.0ms, 6 car, 1 airplane
Speed: 20.3ms track, 64.0ms solution per image at shape (1, 3, 720, 1280)

8: 720x1280 73.8ms, 6 car, 1 airplane
Speed: 40.1ms track, 73.8ms solution per image at shape (1, 3, 720, 12

Фокусировка на объектах

In [14]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error reading video file"


w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("videos/visioneye_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))


visioneye = solutions.VisionEye(
    show=True,  # display the output
    model="yolo_models/yolo11n.pt",  # use any model that Ultralytics support, i.e, YOLOv10
    classes=[0, 2],  # generate visioneye view for specific classes
    vision_point=(50, 50),  # the point, where vision will view objects and draw tracks
)


while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    results = visioneye(im0)

    print(results)  # access the output

    video_writer.write(results.plot_im)

cap.release()
video_writer.release()
cv2.destroyAllWindows()

Ultralytics Solutions:  {'source': None, 'model': 'yolo_models/yolo11n.pt', 'classes': [0, 2], 'show_conf': True, 'show_labels': True, 'region': None, 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (50, 50), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': True, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'half': False, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}
0: 720x1280 25.2ms, 6 car
Speed: 107.7ms track, 25.2ms solution per image at shape (1, 3, 720, 1280)

total_tracks=6, speed={'track': 107.7471999451518, 'solution': 25.245700031518936}
1: 720x1280 2.7ms, 6 car
Speed: 29.8ms track, 2.7ms solution per image at shape (1, 3, 720, 1280)

total_tracks=6, speed={'track': 29.757200041785836, 'solut

Аналитика

In [15]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error reading video file"


w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
out = cv2.VideoWriter(
    "videos/analytics_output.avi",
    cv2.VideoWriter_fourcc(*"MJPG"),
    fps,
    (1280, 720),  # this is fixed
)


analytics = solutions.Analytics(
    show=True,  # display the output
    analytics_type="line",  # pass the analytics type, could be "pie", "bar" or "area".
    model="yolo_models/yolo11n.pt",  # path to the YOLO11 model file
    # classes=[0, 2],  # display analytics for specific detection classes
)


frame_count = 0
while cap.isOpened():
    success, im0 = cap.read()
    if success:
        frame_count += 1
        results = analytics(im0, frame_count)  # update analytics graph every frame

        # print(results)  # access the output

        out.write(results.plot_im)
    else:
        break

cap.release()
out.release()
cv2.destroyAllWindows()

Ultralytics Solutions:  {'source': None, 'model': 'yolo_models/yolo11n.pt', 'classes': None, 'show_conf': True, 'show_labels': True, 'region': None, 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': True, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'half': False, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}
0: 720x1280 59.4ms, 6 car, 1 airplane
Speed: 73.3ms track, 59.4ms solution per image at shape (1, 3, 720, 1280)

1: 720x1280 0.1ms, 6 car, 1 airplane
Speed: 38.5ms track, 0.1ms solution per image at shape (1, 3, 720, 1280)

2: 720x1280 0.1ms, 6 car, 1 airplane
Speed: 40.9ms track, 0.1ms solution per image at shape (1, 3, 720, 1280)

3: 720x12

Задача: реализовать Трекинг объектов с сохранением в видео